# Listar usuarios

Muestra **todos los usuarios guardados**, ordenados por id.

**Qué pasa en la base:** nada, solo lee (⚠️ **usa la base de datos real** (`src/db/users.sqlite`)).

**Acción que usa:** `GET http://127.0.0.1:3004/users`. No recibe datos. Responde `200` con la lista, vacía si no hay usuarios.

**Anotaciones:** necesitas `pnpm start` y `pnpm node-http` en marcha. Pulsa **▶ Run** (o `Shift + Enter`). **Cómo reconocer que ha funcionado:** sale una tabla con una fila por usuario. Si no hubiera ninguno, solo se ven los encabezados.

- `The database failed.` (500): el programa de la base de datos no responde o SQLite falla. Arranca `pnpm start` y repite.
- **Con la base parada:** para el programa de la base de datos (`pnpm start`) y vuelve a pulsar **▶ Run**: sale `The database failed.` y ninguna tabla.
- Si sale `The HTTP client is not running. Start it with pnpm node-http.`, arranca el cliente en otra terminal (con `pnpm start` ya en marcha) y repite.

[← Anterior: Crear un usuario](create-user.ipynb) · [Siguiente: Buscar por id →](find-user-by-id.ipynb)

In [ ]:
import json
import os
import subprocess
import urllib.error
import urllib.request

URL = "http://127.0.0.1:3004/users"
request = urllib.request.Request(URL, method="GET")

# Envía la petición al cliente. Una respuesta de error (400, 404, 409, 500) también trae un texto
status = None
text = ""
try:
    with urllib.request.urlopen(request) as answer:
        status = answer.status
        text = answer.read().decode()
except urllib.error.HTTPError as error:
    status = error.code
    text = error.read().decode()
except urllib.error.URLError:
    pass

# Código de Node que dibuja las filas en una tabla con cli-table3
js = """
import Table from "cli-table3";

const rows = JSON.parse(process.argv[1]);

const columns = ["id", "name", "email", "createdAt", "updatedAt"];
const table = new Table({ head: columns, style: { head: ["cyan", "bold"] } });

for (const row of rows) {
  table.push(columns.map((column) => row[column]));
}
console.log(table.toString());
"""

if status is None:
    print("The HTTP client is not running. Start it with pnpm node-http.")
elif status >= 400:
    # Un fallo: solo el mensaje, sin tabla
    print(json.loads(text)["error"])
else:
    data = json.loads(text)
    rows = data
    # Lanza Node desde la carpeta first-app (tres carpetas más arriba) y le pasa las filas como JSON
    result = subprocess.run(
        ["node", "--disable-warning=ExperimentalWarning", "--input-type=module", "-e", js, json.dumps(rows)],
        cwd="../../..",
        env={**os.environ, "FORCE_COLOR": "1"},  # conserva el color de los encabezados
        capture_output=True,
        text=True,
    )
    print(result.stdout or result.stderr)